# Tutorial 4 — CLR sensitivity analysis

**Goal:** assess whether conclusions from mapped cell-type ratios depend on compositional closure. Raw mapped ratios remain the primary interpretable representation; centered log-ratio (CLR) results are a sensitivity analysis.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

HERE = Path.cwd() if (Path.cwd() / 'tutorial_utils.py').exists() else Path.cwd() / 'tutorials'
sys.path.insert(0, str(HERE))
from tutorial_utils import find_repo_root, load_prepared_or_example, align_and_validate
from HomoloMap.stats import SpinTest
from HomoloMap.utils import run_spin_correlations

ROOT = find_repo_root()
N_SPINS = 100
SEED = 42
X, Y = load_prepared_or_example(ROOT, level='subclass')
X, Y = align_and_validate(X, Y, require_complete_bn=True)
spinner = SpinTest(atlas='BN', n_spins=N_SPINS, seed=SEED)

In [ ]:
ratio = run_spin_correlations(
    X, Y, spinner, metric='pearsonr', FDR='fdr_bh', n_jobs=1,
    composition_transform='none',
)
clr = run_spin_correlations(
    X, Y, spinner, metric='pearsonr', FDR='fdr_bh', n_jobs=1,
    composition_transform='clr',
    composition_params={'zero_method': 'multiplicative'},
)

In [ ]:
idp = Y.columns[0]
r_col = f'{idp}_ratio_spin_r'
comparison = pd.DataFrame({'ratio_r': ratio[r_col], 'clr_r': clr[r_col]})
print('Across-cell-type agreement')
display(comparison.corr())
display(comparison.reindex(comparison.ratio_r.abs().sort_values(ascending=False).index).head(10))

## Interpretation

CLR effects are relative log-contrasts, not abundance effects. Compare effect direction, rank, and inferential conclusions rather than expecting identical coefficients. Do not apply CLR to density values without a defensible compositional definition.

In [ ]:
OUTPUT = ROOT / 'tutorial_outputs'
OUTPUT.mkdir(exist_ok=True)
ratio.to_csv(OUTPUT / 'spin_results_ratio.csv')
clr.to_csv(OUTPUT / 'spin_results_clr_sensitivity.csv')
comparison.to_csv(OUTPUT / 'ratio_clr_effect_comparison.csv')